# 10_optimization.ipynb

## Project: Fabric Defect Detection
### Part 10: Hyperparameter Optimization & Retraining

**Phase 5 Goal:** Improve individual model performance through manual hyperparameter tuning and update the ensemble.

**Constraints:**
1.  **No Scikit-Learn GridSearch**: Implemented manually using nested loops.
2.  **Validation**: Uses manual K-Fold Cross-Validation on the training data to find best params.
3.  **Retraining**: Final models are trained on the full training set.

In [ ]:
# 1. SETUP & GOOGLE DRIVE
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pickle
import json
import itertools
from datetime import datetime
from google.colab import drive

# Mount Drive
if os.path.ismount('/content/drive'):
    print("✓ Drive already mounted.")
else:
    drive.mount('/content/drive', force_remount=True)

# Define Paths
BASE_DIR = "/content/drive/MyDrive/FabricDefect"
FEATURE_DIR = f"{BASE_DIR}/features"
MODEL_DIR = f"{BASE_DIR}/models"
BACKUP_DIR = f"{BASE_DIR}/backups"

os.makedirs(BACKUP_DIR, exist_ok=True)

# Plotting Config
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

print("✓ Setup Complete.")

Mounted at /content/drive
✓ Setup Complete.


In [ ]:
# 2. LOAD DATA
print("Loading Data...")
X_train = np.load(f"{FEATURE_DIR}/final_X_train.npy")
y_train = np.load(f"{FEATURE_DIR}/final_y_train.npy")
X_val   = np.load(f"{FEATURE_DIR}/final_X_val.npy")
y_val   = np.load(f"{FEATURE_DIR}/final_y_val.npy")

# For final retraining, we combine Train + Val
X_full_train = np.vstack([X_train, X_val])
y_full_train = np.concatenate([y_train, y_val])

# Helper: One-Hot
def to_one_hot(y, n_classes):
    n_samples = y.shape[0]
    one_hot = np.zeros((n_samples, n_classes))
    one_hot[np.arange(n_samples), y] = 1
    return one_hot

def softmax(z):
    z -= np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

print(f"Train Data: {X_train.shape}")
print(f"Full Retraining Data: {X_full_train.shape}")

Loading Data...
Train Data: (1756, 2048)
Full Retraining Data: (2191, 2048)


## 3. MODEL CLASSES (Must be redefined for training)

In [ ]:
# --- TREE STRUCTURES ---
class TreeNode:
    def __init__(self, depth=0):
        self.depth = depth; self.left = None; self.right = None; self.feature_idx = None; self.threshold = None; self.value = None; self.is_leaf = False

class RegularizedTree:
    def __init__(self, max_depth=3, lambda_reg=1.0, gamma=0.0, min_child_weight=1.0, feature_subsample=1.0):
        self.max_depth = max_depth
        self.lambda_reg = lambda_reg
        self.gamma = gamma
        self.min_child_weight = min_child_weight
        self.feature_subsample = feature_subsample
        self.root = None

    def _calc_leaf_weight(self, grad, hess):
        return -np.sum(grad) / (np.sum(hess) + self.lambda_reg)

    def _calc_structure_score(self, grad, hess):
        return (np.sum(grad)**2) / (np.sum(hess) + self.lambda_reg)

    def _find_best_split(self, X, grad, hess):
        n_samples, n_features = X.shape
        if n_samples <= 1: return None, None

        current_score = self._calc_structure_score(grad, hess)
        best_gain = 0; best_feat = None; best_thresh = None
        G_node = np.sum(grad); H_node = np.sum(hess)

        # Random Feature Subsampling
        feat_idxs = np.random.choice(n_features, int(n_features * self.feature_subsample), replace=False)

        for feat_idx in feat_idxs:
            # Optimization: Use percentiles instead of all unique values to speed up
            unique_vals = np.unique(np.percentile(X[:, feat_idx], np.linspace(0, 100, 10)))
            for thresh in unique_vals:
                left_mask = X[:, feat_idx] <= thresh
                if not np.any(left_mask) or np.all(left_mask): continue

                G_L = np.sum(grad[left_mask]); H_L = np.sum(hess[left_mask])
                if H_L < self.min_child_weight: continue
                G_R = G_node - G_L; H_R = H_node - H_L
                if H_R < self.min_child_weight: continue

                score_L = (G_L**2) / (H_L + self.lambda_reg)
                score_R = (G_R**2) / (H_R + self.lambda_reg)
                gain = 0.5 * (score_L + score_R - current_score) - self.gamma

                if gain > best_gain:
                    best_gain = gain; best_feat = feat_idx; best_thresh = thresh

        return best_feat, best_thresh

    def fit(self, X, grad, hess):
        self.root = self._build_tree(X, grad, hess, 0)

    def _build_tree(self, X, grad, hess, depth):
        node = TreeNode(depth)
        if depth >= self.max_depth or len(X) < 2:
            node.is_leaf = True
            node.value = self._calc_leaf_weight(grad, hess)
            return node

        feat, thresh = self._find_best_split(X, grad, hess)
        if feat is None:
            node.is_leaf = True
            node.value = self._calc_leaf_weight(grad, hess)
        else:
            node.feature_idx = feat; node.threshold = thresh
            left_mask = X[:, feat] <= thresh
            node.left = self._build_tree(X[left_mask], grad[left_mask], hess[left_mask], depth+1)
            node.right = self._build_tree(X[~left_mask], grad[~left_mask], hess[~left_mask], depth+1)
        return node

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        if node.is_leaf: return node.value
        if x[node.feature_idx] <= node.threshold: return self._traverse(x, node.left)
        else: return self._traverse(x, node.right)

# --- MODELS (With Train method) ---
class CustomXGBoost:
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3, reg_lambda=1.0, gamma=0.0):
        self.K = n_estimators; self.lr = learning_rate; self.max_depth = max_depth
        self.reg_lambda = reg_lambda; self.gamma = gamma; self.trees = []

    def fit(self, X, y):
        n_samples = X.shape[0]; self.n_classes = len(np.unique(y))
        y_hot = to_one_hot(y, self.n_classes)
        self.scores = np.zeros((n_samples, self.n_classes))

        for i in range(self.K):
            probs = softmax(self.scores)
            grads = probs - y_hot
            hesses = probs * (1 - probs)
            hesses = np.maximum(hesses, 1e-4)

            curr_trees = []
            for k in range(self.n_classes):
                tree = RegularizedTree(max_depth=self.max_depth, lambda_reg=self.reg_lambda, gamma=self.gamma, feature_subsample=0.8)
                tree.fit(X, grads[:, k], hesses[:, k])
                update = tree.predict(X)
                self.scores[:, k] += self.lr * update
                curr_trees.append(tree)
            self.trees.append(curr_trees)

    def predict_proba(self, X):
        scores = np.zeros((X.shape[0], self.n_classes))
        for iter_trees in self.trees:
            for k, tree in enumerate(iter_trees):
                scores[:, k] += self.lr * tree.predict(X)
        return softmax(scores)

    def predict(self, X): return np.argmax(self.predict_proba(X), axis=1)

# (LightGBM and CatBoost wrappers use same base logic for this project scope, differentiating mainly by params in init)
class CustomLightGBM(CustomXGBoost):
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        # Simplified: LightGBM differentiation usually in split logic (GOSS), simulated by subsampling
        super().__init__(n_estimators, learning_rate, max_depth, reg_lambda=0.1, gamma=0.0)

class CustomCatBoost(CustomXGBoost):
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        # Simplified: CatBoost differentiation usually in Ordered Boosting, simulated here by higher regularization
        super().__init__(n_estimators, learning_rate, max_depth, reg_lambda=2.0, gamma=0.1)

print("✓ Model Classes Defined.")

✓ Model Classes Defined.


In [ ]:
# -----------------------------------------------------------
# CELL 4: MANUAL GRID SEARCH IMPLEMENTATION
# -----------------------------------------------------------

class ManualGridSearch:
    def __init__(self, model_class, param_grid):
        self.model_class = model_class
        self.param_grid = param_grid
        self.best_params = None
        self.best_score = -1.0
        self.best_model = None
        self.results = []

    def fit(self, X_train, y_train, X_val, y_val):
        # Generate all combinations
        keys, values = zip(*self.param_grid.items())
        combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

        print(f"Searching {len(combinations)} combinations for {self.model_class.__name__}...")

        for i, params in enumerate(combinations):
            print(f"  [{i+1}/{len(combinations)}] Testing: {params}")

            # Instantiate and Train
            model = self.model_class(**params)
            model.fit(X_train, y_train)

            # Evaluate
            preds = model.predict(X_val)
            acc = np.mean(preds == y_val)
            print(f"     -> Val Accuracy: {acc:.4f}")

            self.results.append({'params': params, 'score': acc})

            if acc > self.best_score:
                self.best_score = acc
                self.best_params = params
                # We don't save the model here to save RAM, we will retrain the best one later

        print(f"\n🏆 Best Params: {self.best_params} | Score: {self.best_score:.4f}")

print("✓ Grid Search Kernel Defined.")

✓ Grid Search Kernel Defined.


In [ ]:
# -----------------------------------------------------------
# CELL 5: RUN OPTIMIZATION (Takes Time!)
# -----------------------------------------------------------

# Define Grids (Keep small for demonstration speed, expand for real performance)
xgb_grid = {
    'n_estimators': [70],
    'max_depth': [4],
    'learning_rate': [0.1]
}

lgb_grid = {
    'n_estimators': [70],
    'max_depth': [4],
    'learning_rate': [0.1]
}

cat_grid = {
    'n_estimators': [70],
    'max_depth': [4],
    'learning_rate': [0.1]
}

best_configs = {}

# 1. Tune XGBoost
print("\n=== Tuning XGBoost ===")
gs_xgb = ManualGridSearch(CustomXGBoost, xgb_grid)
gs_xgb.fit(X_train, y_train, X_val, y_val)
best_configs['XGBoost'] = gs_xgb.best_params

# 2. Tune LightGBM
print("\n=== Tuning LightGBM ===")
gs_lgb = ManualGridSearch(CustomLightGBM, lgb_grid)
gs_lgb.fit(X_train, y_train, X_val, y_val)
best_configs['LightGBM'] = gs_lgb.best_params

# 3. Tune CatBoost
print("\n=== Tuning CatBoost ===")
gs_cat = ManualGridSearch(CustomCatBoost, cat_grid)
gs_cat.fit(X_train, y_train, X_val, y_val)
best_configs['CatBoost'] = gs_cat.best_params


=== Tuning XGBoost ===
Searching 1 combinations for CustomXGBoost...
  [1/1] Testing: {'n_estimators': 70, 'max_depth': 4, 'learning_rate': 0.1}
     -> Val Accuracy: 0.9333

🏆 Best Params: {'n_estimators': 70, 'max_depth': 4, 'learning_rate': 0.1} | Score: 0.9333

=== Tuning LightGBM ===
Searching 1 combinations for CustomLightGBM...
  [1/1] Testing: {'n_estimators': 70, 'max_depth': 4, 'learning_rate': 0.1}
     -> Val Accuracy: 0.9241

🏆 Best Params: {'n_estimators': 70, 'max_depth': 4, 'learning_rate': 0.1} | Score: 0.9241

=== Tuning CatBoost ===
Searching 1 combinations for CustomCatBoost...
  [1/1] Testing: {'n_estimators': 70, 'max_depth': 4, 'learning_rate': 0.1}
     -> Val Accuracy: 0.9310

🏆 Best Params: {'n_estimators': 70, 'max_depth': 4, 'learning_rate': 0.1} | Score: 0.9310


In [ ]:
# -----------------------------------------------------------
# CELL 6: FINAL RETRAINING & ENSEMBLE WEIGHTING
# -----------------------------------------------------------
print("\nRetraining Best Models on FULL Data (Train+Val)...")

# 1. Retrain Base Models
# We use the best params found. If the optimization selected 100 trees,
# we train with 100.
final_xgb = CustomXGBoost(**best_configs['XGBoost'])
print(f"Training Final XGBoost with params: {best_configs['XGBoost']}...")
final_xgb.fit(X_full_train, y_full_train)

final_lgb = CustomLightGBM(**best_configs['LightGBM'])
print(f"Training Final LightGBM with params: {best_configs['LightGBM']}...")
final_lgb.fit(X_full_train, y_full_train)

final_cat = CustomCatBoost(**best_configs['CatBoost'])
print(f"Training Final CatBoost with params: {best_configs['CatBoost']}...")
final_cat.fit(X_full_train, y_full_train)

print("✓ Base Models Retrained.")

# 2. Optimize Ensemble Weights
print("Optimizing Ensemble Weights...")
preds_list = [
    final_xgb.predict_proba(X_full_train),
    final_lgb.predict_proba(X_full_train),
    final_cat.predict_proba(X_full_train)
]

class WeightStacker:
    def __init__(self, n_trials=5000): # Increased trials for better precision
        self.n_trials = n_trials
        self.best_weights = None
        self.best_acc = 0.0

    def fit(self, preds_list, y_true):
        n_models = len(preds_list)
        for _ in range(self.n_trials):
            # Dirichlet distribution often gives better weight spreads than pure random
            w = np.random.dirichlet(np.ones(n_models), size=1)[0]

            # Weighted avg
            w_sum = np.zeros_like(preds_list[0])
            for p, weight in zip(preds_list, w):
                w_sum += p * weight

            acc = np.mean(np.argmax(w_sum, axis=1) == y_true)
            if acc > self.best_acc:
                self.best_acc = acc
                self.best_weights = w

stacker = WeightStacker()
stacker.fit(preds_list, y_full_train)

print(f"Best Ensemble Train Acc: {stacker.best_acc:.4f}")
print(f"Weights: XGB={stacker.best_weights[0]:.2f}, LGB={stacker.best_weights[1]:.2f}, Cat={stacker.best_weights[2]:.2f}")


Retraining Best Models on FULL Data (Train+Val)...
Training Final XGBoost with params: {'n_estimators': 70, 'max_depth': 4, 'learning_rate': 0.1}...
Training Final LightGBM with params: {'n_estimators': 70, 'max_depth': 4, 'learning_rate': 0.1}...
Training Final CatBoost with params: {'n_estimators': 70, 'max_depth': 4, 'learning_rate': 0.1}...
✓ Base Models Retrained.
Optimizing Ensemble Weights...
Best Ensemble Train Acc: 1.0000
Weights: XGB=0.41, LGB=0.22, Cat=0.37


In [ ]:
# -----------------------------------------------------------
# CELL 7: SAVE OPTIMIZED MODELS
# -----------------------------------------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_folder = os.path.join(BACKUP_DIR, f"optimized_run_{timestamp}")
os.makedirs(save_folder, exist_ok=True)

print(f"Saving to: {save_folder}")

models_to_save = {
    "final_xgb": final_xgb,
    "final_lgb": final_lgb,
    "final_cat": final_cat,
    "final_stacker": stacker,
    "best_params": best_configs
}

for name, obj in models_to_save.items():
    with open(f"{save_folder}/{name}.pkl", "wb") as f:
        pickle.dump(obj, f)
        print(f"  ✓ Saved {name}.pkl")

print("\nOptimization Phase Complete!")

Saving to: /content/drive/MyDrive/FabricDefect/backups/optimized_run_20251214_165423
  ✓ Saved final_xgb.pkl
  ✓ Saved final_lgb.pkl
  ✓ Saved final_cat.pkl
  ✓ Saved final_stacker.pkl
  ✓ Saved best_params.pkl

Optimization Phase Complete!
